# 🛠️ Notebook 2: LinkedIn (professional network) - Implementation

In Notebook 1 we picked the "best" design choices. Now we assemble them into a working mini-LinkedIn:

- Profiles with experience, education, and skills
- Connection requests with a guarded state machine
- Companies, jobs, and applications (also with a state machine)
- Endorsements as (endorser, endorsee, skill) edges
- A simple Messaging inbox restricted to connections
- A tiny JobRecommender that ranks jobs by skill match and "people in company you know"

Every step is runnable, with asserts and prints to prove the behavior.

## 🛠️ Setup

```bash
cd 07-object-oriented-design/linkedin
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` -> **Reload Window**.

> This notebook assumes you have read **Notebook 1** - we are now putting the "best" choices from that notebook into one working implementation.

## Step 1 - Imports, IDs, and `Profile` building blocks

- `dataclass` saves us from writing `__init__` / `__repr__` by hand.
- `field(default_factory=...)` is the correct pattern for mutable defaults (never `= []` or `= {}`).
- `itertools.count` gives us simple stable integer IDs without a database.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from itertools import count
from typing import Optional

# Stable id generators (one per entity type). In a real system these come from the DB.
_uid = count(1)   # users
_jid = count(1)   # jobs
_rid = count(1)   # connection requests
_aid = count(1)   # applications
_mid = count(1)   # messages

@dataclass
class Experience:
    title: str
    company: str
    start_year: int
    end_year: Optional[int] = None   # None = current

    def is_current(self) -> bool:
        return self.end_year is None

@dataclass
class Education:
    school: str
    degree: str
    year: int

@dataclass
class Profile:
    headline: str = ""
    experiences: list[Experience] = field(default_factory=list)
    education:   list[Education]  = field(default_factory=list)
    skills:      set[str]         = field(default_factory=set)

    def add_skill(self, s: str) -> None:
        self.skills.add(s.lower())   # normalize so 'Python' == 'python'

p = Profile(headline="Backend engineer")
p.add_skill("Python"); p.add_skill("python"); p.add_skill("SQL")
p.experiences.append(Experience("Engineer", "Foo Inc", 2020, 2024))
p.experiences.append(Experience("Senior Engineer", "Bar Corp", 2024))   # current
print(p.skills)                            # {'python','sql'} (deduped + lowered)
print("current roles:", [e.title for e in p.experiences if e.is_current()])

## Step 2 - `User` with a hashable identity

Dataclasses with mutable fields are unhashable by default. We hash by the stable integer `id` so a `User` can live in a `set`
(needed for `connections: set[User]`) and survive profile edits without changing identity.

In [ ]:
@dataclass
class User:
    name: str
    id: int = field(default_factory=lambda: next(_uid))
    profile: Profile = field(default_factory=Profile)
    connections: set["User"] = field(default_factory=set)

    def __hash__(self): return self.id
    def __eq__(self, o): return isinstance(o, User) and self.id == o.id
    def __repr__(self): return f"User({self.name})"

alice = User("Alice"); alice.profile.headline = "Backend eng"
for s in ("python", "sql", "redis"): alice.profile.add_skill(s)

bob   = User("Bob");   bob.profile.add_skill("python"); bob.profile.add_skill("go")
carol = User("Carol"); carol.profile.add_skill("sql");  carol.profile.add_skill("kafka")

print(alice, bob, carol)
print("alice id:", alice.id)

## Step 3 - `ConnectionRequest` with a guarded state machine

The state machine (from Notebook 1):

```
PENDING --accept()--> ACCEPTED    (terminal)
PENDING --reject()--> REJECTED    (terminal)
```

- `accept()` and `reject()` raise if called from a non-PENDING state - this prevents double-accept bugs.
- On accept we mirror the edge into both users' `connections` sets so the relationship is symmetric.

In [ ]:
class ReqStatus(Enum):
    PENDING  = "pending"
    ACCEPTED = "accepted"
    REJECTED = "rejected"

@dataclass
class ConnectionRequest:
    sender: User
    receiver: User
    id: int = field(default_factory=lambda: next(_rid))
    status: ReqStatus = ReqStatus.PENDING

    def accept(self) -> None:
        if self.status is not ReqStatus.PENDING:
            raise ValueError(f"cannot accept from {self.status}")
        self.status = ReqStatus.ACCEPTED
        self.sender.connections.add(self.receiver)
        self.receiver.connections.add(self.sender)

    def reject(self) -> None:
        if self.status is not ReqStatus.PENDING:
            raise ValueError(f"cannot reject from {self.status}")
        self.status = ReqStatus.REJECTED

r1 = ConnectionRequest(alice, bob); r1.accept()
r2 = ConnectionRequest(alice, carol); r2.reject()

assert bob in alice.connections and alice in bob.connections
assert carol not in alice.connections              # rejected -> no edge
print("alice connections:", alice.connections)
try:
    r1.accept()                                    # already accepted
except ValueError as e:
    print("guard works:", e)

## Step 4 - `ConnectionService`: one outstanding request per pair

A subtle real-world rule: you shouldn't be able to send a *second* pending request while the first is still pending,
and you shouldn't send one at all to someone you're already connected to. We encapsulate those rules in a tiny service
so the `ConnectionRequest` class stays small.

In [ ]:
class ConnectionService:
    def __init__(self):
        self._requests: list[ConnectionRequest] = []

    def _pair_key(self, a: User, b: User) -> frozenset:
        return frozenset({a.id, b.id})

    def pending_between(self, a: User, b: User) -> Optional[ConnectionRequest]:
        for r in self._requests:
            if r.status is ReqStatus.PENDING and self._pair_key(r.sender, r.receiver) == self._pair_key(a, b):
                return r
        return None

    def send(self, sender: User, receiver: User) -> ConnectionRequest:
        if sender == receiver:
            raise ValueError("can't connect to yourself")
        if receiver in sender.connections:
            raise ValueError("already connected")
        if self.pending_between(sender, receiver) is not None:
            raise ValueError("a pending request already exists between these users")
        r = ConnectionRequest(sender, receiver)
        self._requests.append(r)
        return r

cs = ConnectionService()
dave = User("Dave"); eve = User("Eve")

cs.send(dave, eve)
for case in ("same user again", "reverse direction", "already connected"):
    try:
        if case == "same user again":
            cs.send(dave, eve)
        elif case == "reverse direction":
            cs.send(eve, dave)
        else:
            cs.pending_between(dave, eve).accept()
            cs.send(dave, eve)
    except ValueError as e:
        print(case, "->", e)

## Step 5 - Companies, Jobs, and Applications

Companies post jobs with a set of `required_skills`. An `Application` has its own state machine:

```
SUBMITTED --review()--> REVIEWED --make_offer()--> OFFER     (terminal)
                                `--reject()-----> REJECTED  (terminal)
SUBMITTED --reject()---> REJECTED  (terminal)
```

In [ ]:
@dataclass
class Company:
    name: str
    employees: list[User] = field(default_factory=list)

    def hire(self, u: User) -> None:
        if u not in self.employees:
            self.employees.append(u)

@dataclass
class Job:
    title: str
    company: Company
    description: str
    required_skills: set[str] = field(default_factory=set)
    id: int = field(default_factory=lambda: next(_jid))

    def match_score(self, profile: Profile) -> float:
        """How much of this job's requirement list does a profile cover? 0..1

        This lives on Job, not Application, on purpose. "How well does this
        person fit this role?" is a question you ask BEFORE anyone applies —
        search, recommendations, and 'jobs you may like' all need it. Hanging it
        off Application would force those callers to fabricate an Application
        (burning an id, implying a submission that never happened) just to do
        arithmetic. Rule of thumb: put a calculation on the object that owns the
        data it reads, never on a record of an event that has not occurred.
        """
        if not self.required_skills:
            return 1.0                       # no requirements -> full match
        return len(profile.skills & self.required_skills) / len(self.required_skills)

class AppStatus(Enum):
    SUBMITTED = "submitted"
    REVIEWED  = "reviewed"
    OFFER     = "offer"
    REJECTED  = "rejected"

@dataclass
class Application:
    user: User
    job: Job
    id: int = field(default_factory=lambda: next(_aid))
    status: AppStatus = AppStatus.SUBMITTED

    # Guarded transitions
    def review(self) -> None:
        if self.status is not AppStatus.SUBMITTED:
            raise ValueError(f"cannot review from {self.status}")
        self.status = AppStatus.REVIEWED

    def make_offer(self) -> None:
        if self.status is not AppStatus.REVIEWED:
            raise ValueError(f"offers only from REVIEWED, not {self.status}")
        self.status = AppStatus.OFFER

    def reject(self) -> None:
        if self.status in (AppStatus.OFFER, AppStatus.REJECTED):
            raise ValueError(f"cannot reject from {self.status}")
        self.status = AppStatus.REJECTED

    def match_score(self) -> float:
        # Thin delegation: one implementation of the rule, two convenient callers.
        return self.job.match_score(self.user.profile)

foo = Company("Foo Inc"); foo.hire(alice)
job = Job("Senior Backend", foo, "Distributed systems",
          required_skills={"python", "sql", "kafka"})

a1 = Application(alice, job)   # has python+sql -> 2/3
a2 = Application(bob,   job)   # has python     -> 1/3
print(f"alice match: {a1.match_score():.2f}")
print(f"bob   match: {a2.match_score():.2f}")

a1.review(); a1.make_offer()
try: a1.make_offer()                              # already OFFER
except ValueError as e: print("guard:", e)

## Step 6 - Endorsements as graph edges

One `EndorsementBook` stores `(endorser, endorsee, skill)` tuples in a set. Duplicates are deduped automatically,
and we can ask "who endorsed Alice for Python?" or "how many endorsements does Alice have for Python?" cheaply.

In [ ]:
@dataclass(frozen=True)
class Endorsement:
    endorser_id: int
    endorsee_id: int
    skill: str

class EndorsementBook:
    def __init__(self):
        self._edges: set[Endorsement] = set()

    def endorse(self, endorser: User, endorsee: User, skill: str) -> None:
        if endorser == endorsee:
            raise ValueError("can't endorse yourself")
        if skill.lower() not in endorsee.profile.skills:
            raise ValueError(f"{endorsee.name} doesn't list '{skill}' as a skill")
        self._edges.add(Endorsement(endorser.id, endorsee.id, skill.lower()))

    def endorsers_of(self, user: User, skill: str) -> list[int]:
        s = skill.lower()
        return [e.endorser_id for e in self._edges if e.endorsee_id == user.id and e.skill == s]

    def count(self, user: User, skill: str) -> int:
        return len(self.endorsers_of(user, skill))

eb = EndorsementBook()
eb.endorse(bob,   alice, "python")
eb.endorse(bob,   alice, "python")    # dedup -> still 1
eb.endorse(carol, alice, "python")
print("alice python endorsements:", eb.count(alice, "python"))   # 2
try:
    eb.endorse(bob, alice, "rust")    # alice doesn't list rust
except ValueError as e:
    print("guard:", e)

## Step 7 - Messaging (DMs) restricted to connections

LinkedIn lets you DM your connections for free; messaging strangers requires InMail (a premium feature).
We encode the basic rule here: messaging a non-connection raises. Each user has an `Inbox` that stores messages received.

In [ ]:
@dataclass
class Message:
    sender: User
    receiver: User
    text: str
    ts: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    id: int = field(default_factory=lambda: next(_mid))

class Inbox:
    def __init__(self):
        self._messages: list[Message] = []

    def deliver(self, m: Message) -> None:
        self._messages.append(m)

    def latest(self, n: int = 5) -> list[Message]:
        return self._messages[-n:]

# Attach an Inbox to each user lazily (could also make it a field on User)
inboxes: dict[int, Inbox] = {}

def inbox_of(u: User) -> Inbox:
    if u.id not in inboxes:
        inboxes[u.id] = Inbox()
    return inboxes[u.id]

def send_dm(sender: User, receiver: User, text: str) -> Message:
    if receiver not in sender.connections:
        raise PermissionError(f"{sender.name} is not connected to {receiver.name} - use InMail")
    m = Message(sender, receiver, text)
    inbox_of(receiver).deliver(m)
    return m

send_dm(alice, bob, "hey, saw you also do Python")
send_dm(bob, alice, "yes! currently learning Rust on the side")

for m in inbox_of(alice).latest():
    print(f"[{m.sender.name} -> {m.receiver.name}] {m.text}")

try:
    send_dm(alice, carol, "hello")   # not connected
except PermissionError as e:
    print("blocked:", e)

## Step 8 - A tiny `JobRecommender`

Two signals, combined:

1. **Skill match** - the `match_score` we already have (0..1).
2. **"You know someone at the company"** - a small bonus if any of the user's connections work there.

The final score is a weighted sum. This mirrors how real job-matching systems combine multiple features.

In [ ]:
class JobRecommender:
    def __init__(self, jobs: list[Job], skill_weight: float = 0.8, network_weight: float = 0.2):
        self.jobs = jobs
        self.sw = skill_weight
        self.nw = network_weight

    def score(self, user: User, job: Job) -> float:
        # Ask the Job directly. No Application is created, so recommending a job
        # never leaves a phantom application behind.
        skill = job.match_score(user.profile)
        network = 1.0 if any(c in job.company.employees for c in user.connections) else 0.0
        return self.sw * skill + self.nw * network

    def recommend(self, user: User, top_k: int = 3) -> list[tuple[Job, float]]:
        ranked = [(j, self.score(user, j)) for j in self.jobs]
        ranked.sort(key=lambda pair: pair[1], reverse=True)
        return ranked[:top_k]

bar = Company("Bar Corp"); bar.hire(carol)
jobs = [
    Job("Senior Backend",   foo, "distributed", required_skills={"python", "sql", "kafka"}),
    Job("Data Engineer",    bar, "pipelines",   required_skills={"sql", "kafka"}),
    Job("Frontend Engineer", foo, "react",      required_skills={"javascript", "react"}),
]

# Make sure Bob is connected to Alice (at Foo) and Carol (at Bar) so the network bonus shows up
ConnectionRequest(bob, carol).accept()

rec = JobRecommender(jobs)
for job, s in rec.recommend(bob):
    bonus = " (+network)" if any(c in job.company.employees for c in bob.connections) else ""
    print(f"{s:.2f}  {job.title:<18} @ {job.company.name}{bonus}")

## Step 9 - Putting it all together: an end-to-end scenario

We simulate a realistic flow:

1. Alice and Bob create profiles with skills.
2. Bob sends Alice a connection request; Alice accepts.
3. Alice endorses Bob for Python.
4. Foo Inc. posts a job; Bob applies; HR reviews and makes an offer.
5. Alice DMs Bob to congratulate him.

In [ ]:
# Reset state for a clean scenario (fresh counters -> easy-to-read IDs)
_uid = count(1); _jid = count(1); _rid = count(1); _aid = count(1); _mid = count(1)
inboxes.clear()

alice = User("Alice"); alice.profile.headline = "Backend eng"
for s in ("python", "sql", "redis"): alice.profile.add_skill(s)

bob = User("Bob")
for s in ("python", "sql"): bob.profile.add_skill(s)

# 1. connection
cs = ConnectionService()
req = cs.send(bob, alice); req.accept()
assert alice in bob.connections

# 2. endorsement
eb = EndorsementBook()
eb.endorse(alice, bob, "python")
print("Bob python endorsements:", eb.count(bob, "python"))

# 3. job post + application
foo = Company("Foo Inc"); foo.hire(alice)
job = Job("Senior Backend", foo, "distributed systems",
          required_skills={"python", "sql", "kafka"})
app = Application(bob, job)
print(f"Bob's match: {app.match_score():.2f}")
app.review(); app.make_offer()
assert app.status is AppStatus.OFFER

# 4. DM (connected -> allowed)
send_dm(alice, bob, f"congrats on the offer at {foo.name}!")
print("Bob's inbox:", [f"{m.sender.name}: {m.text}" for m in inbox_of(bob).latest()])

# 5. Recommender - highest-scored job for Bob
rec = JobRecommender([job,
                      Job("Data Eng", foo, "etl", required_skills={"sql"})])
for j, s in rec.recommend(bob, top_k=2):
    print(f"  {s:.2f}  {j.title}")

## Step 10 — Verify the design

The scenario above *prints*; prints only prove that the code ran. These assertions state the
rules themselves, so a later refactor that quietly re-opens a terminal state or lets a stranger
DM you fails immediately and loudly.

In [ ]:
# --- ConnectionRequest is a state machine with TERMINAL states -------------
u1, u2 = User("U1"), User("U2")
r = ConnectionRequest(u1, u2)
assert r.status is ReqStatus.PENDING
r.accept()
assert u2 in u1.connections and u1 in u2.connections, "accept mirrors the edge both ways"
for illegal in ("accept", "reject"):
    try:
        getattr(r, illegal)()
        raise AssertionError(f"{illegal}() from ACCEPTED must raise")
    except ValueError:
        pass

rejected = ConnectionRequest(User("U3"), User("U4"))
rejected.reject()
assert not rejected.sender.connections, "a rejected request creates NO edge"

# --- ConnectionService: no self-connects, no duplicate pending requests ----
svc = ConnectionService()
a, b = User("A"), User("B")
for bad, exc_text in [((a, a), "yourself")]:
    try:
        svc.send(*bad); raise AssertionError("self-connect must raise")
    except ValueError as e:
        assert exc_text in str(e)
svc.send(a, b)
for pair in [(a, b), (b, a)]:                 # duplicate, in both directions
    try:
        svc.send(*pair); raise AssertionError("duplicate pending request must raise")
    except ValueError:
        pass
svc.pending_between(a, b).accept()
assert svc.pending_between(a, b) is None,     "accepting clears the pending request"
try:
    svc.send(a, b); raise AssertionError("already-connected must raise")
except ValueError:
    pass

# --- Skill matching lives on Job and is genuinely side-effect free ---------
co = Company("Acme")
j = Job("Eng", co, "d", required_skills={"python", "sql", "kafka"})
p_ = Profile(); [p_.add_skill(s) for s in ("Python", "SQL", "Rust")]
assert p_.skills == {"python", "sql", "rust"},  "add_skill normalizes case"
assert j.match_score(p_) == 2 / 3
assert Job("Any", co, "d").match_score(Profile()) == 1.0, "no requirements -> full match"

before = next(_aid)                            # snapshot the application counter
JobRecommender([j]).recommend(User("Nobody"))
assert next(_aid) == before + 1,               "recommending must NOT create Applications"

# --- Application state machine: offers only from REVIEWED -----------------
applicant = User("Applicant")
app_ = Application(applicant, j)
try:
    app_.make_offer(); raise AssertionError("cannot offer straight from SUBMITTED")
except ValueError:
    pass
app_.review(); app_.make_offer()
assert app_.status is AppStatus.OFFER
try:
    app_.reject(); raise AssertionError("cannot reject an accepted offer")
except ValueError:
    pass

# --- Endorsements are deduped edges, not counters -------------------------
book = EndorsementBook()
e1, e2 = User("E1"), User("E2")
e2.profile.add_skill("python")
book.endorse(e1, e2, "Python"); book.endorse(e1, e2, "python")   # same edge twice
assert book.count(e2, "python") == 1,          "the same endorser counts once"
assert book.endorsers_of(e2, "python") == [e1.id], "we know WHO endorsed, not just how many"
for bad in [(e2, e2, "python"),                # self-endorsement
            (e1, e2, "cobol")]:                # skill not on the profile
    try:
        book.endorse(*bad); raise AssertionError(f"{bad} must raise")
    except ValueError:
        pass

# --- Messaging is gated on the connection edge ----------------------------
m1, m2 = User("M1"), User("M2")
try:
    send_dm(m1, m2, "hi"); raise AssertionError("stranger DM must be blocked")
except PermissionError:
    pass
ConnectionRequest(m1, m2).accept()
send_dm(m1, m2, "hi")
assert len(inbox_of(m2).latest()) == 1,        "the message lands in the RECEIVER's inbox"
assert len(inbox_of(m1).latest()) == 0,        "and not in the sender's"

print("all design invariants hold ✅")

## 🧪 Try it yourself

- Add a `InMail` class that lets users message *non-connections* but charges a credit per send.
- Extend `JobRecommender` with a **recency** signal (jobs posted in the last week get a boost).
- Add a `Recommendation` (letter of recommendation) between connected users - model it as its own class with a text body.
- Make `Profile.add_skill` reject skills that contain spaces or uppercase letters after normalization - fail fast on bad data.
- Add a `withdraw()` transition to `Application` so a user can pull out before review.

## 📚 Related concepts in this repo

- `07-object-oriented-design/facebook/` - the social-graph counterpart (friendship, reactions, news feed).
- `07-object-oriented-design/oo-analysis-and-design/` - the general OOD method we applied here.
- `04-patterns/state/` (if present) - deeper look at state-machine patterns like our `ReqStatus` transitions.